In [1]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3
# %matplotlib inline
%matplotlib qt5
import mne
mne.viz.set_browser_backend("qt")  # or "matplotlib"
mne.set_config("MNE_BROWSER_BACKEND", "qt")  # or "matplotlib"
%gui qt

import xarray as xr # Assuming you're using this
import numpy as np   # For the example

import xarray as xr
import zarr
import panel as pn
import holoviews as hv
hv.extension('bokeh', logo=False)

import hvplot.xarray
import hvplot.pandas
# This line is crucial for displaying plots in a notebook
hvplot.extension('bokeh') # You can also use 'matplotlib' or 'plotly'

# hv.extension('bokeh')
# hv.extension('matplotlib') # or 'matplotlib'
# hv.extension('plotly') # or 'matplotlib'
from holoviews import opts
import panel as pn
pn.extension()

import IPython

# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

Automatic pdb calling has been turned OFF
Using qt as 2D backend.



# Use MNE to load and analyze saved EEG and Motion recordings


In [2]:
import time
import re
from datetime import datetime, timezone

import uuid
from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
from nptyping import NDArray
from matplotlib import pyplot as plt

from pathlib import Path
import numpy as np
import pandas as pd
from numpy.typing import NDArray

import mne
from mne import set_log_level
from copy import deepcopy
import mne

from mne.io import read_raw

datasets = []
# mne.viz.set_browser_backend("Matplotlib")
mne.viz.set_browser_backend("qt")

from mne_lsl.player import PlayerLSL as Player
from mne_lsl.stream import StreamLSL as Stream
from phopylslhelper.easy_time_sync import EasyTimeSyncParsingMixin, readable_dt_str, from_readable_dt_str

from phoofflineeeganalysis.analysis.MNE_helpers import MNEHelpers
from phoofflineeeganalysis.analysis.historical_data import HistoricalData
from phoofflineeeganalysis.analysis.motion_data import MotionData
from phoofflineeeganalysis.analysis.EEG_data import EEGComputations, EEGData
from phoofflineeeganalysis.analysis.anatomy_and_electrodes import ElectrodeHelper
# from ..EegProcessing import bandpower
# from phoofflineeeganalysis.EegProcessing import analyze_eeg_trends
from phoofflineeeganalysis.EegVisualization import VisHelpers
from phoofflineeeganalysis.analysis.SavedSessionsProcessor import SavedSessionsProcessor, SessionModality, DataModalityType

set_log_level("WARNING")


# db_root_path = Path('/content/drive/MyDrive/Databases').resolve()
db_root_path = Path(r'E:/Dropbox (Personal)/Databases').resolve()
assert db_root_path.exists(), f"'{db_root_path.as_posix()}' does not exist!"

# eeg_recordings_file_path: Path = Path(r'E:/Dropbox (Personal)/Databases/UnparsedData/EmotivEpocX_EEGRecordings/fif').resolve()
# headset_motion_recordings_file_path: Path = Path(r'E:/Dropbox (Personal)/Databases/UnparsedData/EmotivEpocX_EEGRecordings/MOTION_RECORDINGS/fif').resolve()

# assert eeg_recordings_file_path.exists()
# assert headset_motion_recordings_file_path.exists()

eeg_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEpocX_EEGRecordings/fif').resolve()
flutter_eeg_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEEG_FlutterRecordings').resolve()
flutter_motion_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEEG_FlutterRecordings/MOTION_RECORDINGS').resolve()
flutter_GENERIC_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEEG_FlutterRecordings/GENERIC_RECORDINGS').resolve()

headset_motion_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEpocX_EEGRecordings/MOTION_RECORDINGS/fif').resolve()
WhisperVideoTranscripts_LSL_Converted = db_root_path.joinpath('UnparsedData/WhisperVideoTranscripts_LSL_Converted').resolve()
pho_log_to_LSL_recordings_path: Path = db_root_path.joinpath('UnparsedData/PhoLogToLabStreamingLayer_logs').resolve()
## These contain little LSL .fif files with names like: '20250808_062814_log.fif',

eeg_analyzed_parent_export_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed').resolve()
pickled_data_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed/PICKLED_COLLECTION').resolve()
assert pickled_data_path.exists()

lab_recorder_output_path = Path(r"E:\Dropbox (Personal)\Databases\UnparsedData\LabRecorderStudies\sub-P001").resolve()
assert lab_recorder_output_path.exists()


# n_most_recent_sessions_to_preprocess: int = None # None means all sessions
n_most_recent_sessions_to_preprocess: int = 35
# n_most_recent_sessions_to_preprocess: int = 5
# n_most_recent_sessions_to_preprocess: int = 10
# n_most_recent_sessions_to_preprocess = None




# modern_found_EEG_recording_files = HistoricalData.get_recording_files(recordings_dir=lab_recorder_output_path, recordings_extensions=['.xdf'])
modern_found_EEG_recording_files = HistoricalData.get_recording_files(recordings_dir=[lab_recorder_output_path, pho_log_to_LSL_recordings_path], recordings_extensions=['.xdf']) ## both sources
modern_found_EEG_recording_files


most_recent_modern_found_EEG_recording_files: List[Path] = modern_found_EEG_recording_files[:n_most_recent_sessions_to_preprocess]
most_recent_modern_found_EEG_recording_files

'qt'

Using matplotlib as 2D backend.


[WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260123_230032_log.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260123_120409_log.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260122_194014_log.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T194023.711Z_eeg.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T134325.495Z_eeg.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260122_134311_log.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_rMBPPinkDot_2026-01-22T022657.502Z.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260122_023741_log.x

[WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260123_230032_log.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260123_120409_log.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260122_194014_log.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T194023.711Z_eeg.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T134325.495Z_eeg.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260122_134311_log.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_rMBPPinkDot_2026-01-22T022657.502Z.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/PhoLogToLabStreamingLayer_logs/20260122_023741_log.x

In [3]:
most_recent_modern_found_EEG_recording_file_df: pd.DataFrame = HistoricalData.build_file_comparison_df(recording_files=most_recent_modern_found_EEG_recording_files)
most_recent_modern_found_EEG_recording_file_df


# modern_found_EEG_recording_file_df: pd.DataFrame = HistoricalData.build_file_comparison_df(recording_files=modern_found_EEG_recording_files)
# modern_found_EEG_recording_file_df

## OUTPUTS: modern_found_EEG_recording_file_df, modern_found_EEG_recording_files

failed to load file: "E:\Dropbox (Personal)\Databases\UnparsedData\PhoLogToLabStreamingLayer_logs\20260123_230032_log.xdf" with error: 'NoneType' object is not subscriptable. Skipping.
failed to load file: "E:\Dropbox (Personal)\Databases\UnparsedData\PhoLogToLabStreamingLayer_logs\20260122_194014_log.xdf" with error: 'NoneType' object is not subscriptable. Skipping.
failed to load file: "E:\Dropbox (Personal)\Databases\UnparsedData\PhoLogToLabStreamingLayer_logs\20260123_120409_log.xdf" with error: 'NoneType' object is not subscriptable. Skipping.
failed to load file: "E:\Dropbox (Personal)\Databases\UnparsedData\PhoLogToLabStreamingLayer_logs\20260122_134311_log.xdf" with error: 'NoneType' object is not subscriptable. Skipping.
failed to load file: "E:\Dropbox (Personal)\Databases\UnparsedData\PhoLogToLabStreamingLayer_logs\20260122_023741_log.xdf" with error: 'NoneType' object is not subscriptable. Skipping.
failed to load file: "E:\Dropbox (Personal)\Databases\UnparsedData\PhoLogTo

,src_file_name,start_t,src_file,meas_datetime,ctime,size,mtime
0,LabRecorder_Apogee_2026-01-22T194023.711Z_eeg,2026-01-22 19:40:23,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-22 19:40:23+00:00,2026-01-22 19:40:23.712537600,48476019,2026-01-22 20:25:26.173330944
1,LabRecorder_Apogee_2026-01-22T134325.495Z_eeg,2026-01-22 13:43:25,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-22 13:43:25+00:00,2026-01-22 13:43:25.497702400,48249985,2026-01-22 14:28:15.394905088
2,LabRecorder_Apogee_2026-01-22T023809.607Z_eeg,2026-01-22 02:38:09,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-22 02:38:09+00:00,2026-01-22 02:38:09.609662208,8747686,2026-01-22 02:53:27.174798080
3,LabRecorder_rMBPPinkDot_2026-01-22T022657.502Z,2026-01-22 02:26:57,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-22 02:26:57+00:00,2026-01-22 05:18:30.762354944,15192194,2026-01-22 02:53:46.000000000
4,LabRecorder_Apogee_2026-01-21T222456.756Z_eeg,2026-01-21 22:24:56,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-21 22:24:56+00:00,2026-01-21 22:24:56.756452864,75856723,2026-01-21 23:35:22.349929472
5,LabRecorder_Apogee_2026-01-21T222210.952Z_eeg,2026-01-21 22:22:10,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-21 22:22:10+00:00,2026-01-21 22:22:10.953361408,1227082,2026-01-21 22:23:22.686732032
6,LabRecorder_Apogee_2026-01-21T221511.412Z_eeg,2026-01-21 22:15:11,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-21 22:15:11+00:00,2026-01-21 22:15:11.419858432,6921285,2026-01-21 22:21:43.269472768
7,LabRecorder_Apogee_2026-01-21T070140.713Z_eeg,2026-01-21 07:01:40,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-21 07:01:40+00:00,2026-01-21 07:01:40.718356736,33453179,2026-01-21 07:32:46.056764160
8,LabRecorder_Apogee_2026-01-20T014745.931Z_eeg,2026-01-20 01:47:45,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-20 01:47:45+00:00,2026-01-20 01:47:45.934224896,124101249,2026-01-20 03:42:58.426438912
9,LabRecorder_Apogee_2026-01-19T204038.425Z_eeg,2026-01-19 20:40:38,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-19 20:40:38+00:00,2026-01-19 20:40:38.429313280,85159355,2026-01-19 21:59:47.339383040


## Timeline

In [ ]:
import pyphoplacecellanalysis.External.pyqtgraph as pg
# from pypho_timeline.widgets import SimpleTimelineWidget, perform_process_all_streams
# from pypho_timeline.__main__ import PositionTrackDatasource, VideoTrackDatasource, main, main_all_modalities_from_xdf_file_example

from pypho_timeline.timeline_builder import TimelineBuilder

# Create Qt application
app = pg.mkQApp("pyPhoTimelineXDFExample")

builder: TimelineBuilder = TimelineBuilder()

In [ ]:
## INPUTS: most_recent_modern_found_EEG_recording_file_df
most_recent_modern_found_EEG_recording_file_df['src_file'].to_list()
demo_xdf_paths: List[Path] = [Path(v) for v in most_recent_modern_found_EEG_recording_file_df['src_file'].to_list()]
timeline = builder.build_from_xdf_files(xdf_file_paths=demo_xdf_paths) 

# SavedSessionProcessor

In [4]:

sso: SavedSessionsProcessor = SavedSessionsProcessor(eeg_recordings_file_path=eeg_recordings_file_path,
                                                     headset_motion_recordings_file_path=headset_motion_recordings_file_path, WhisperVideoTranscripts_LSL_Converted_file_path=WhisperVideoTranscripts_LSL_Converted, pho_log_to_LSL_recordings_path=pho_log_to_LSL_recordings_path,
                                                    eeg_analyzed_parent_export_path=eeg_analyzed_parent_export_path, 
                                                     n_most_recent_sessions_to_preprocess=n_most_recent_sessions_to_preprocess, 
                                                    # should_load_data=False, should_load_preprocessed=False,
                                                    should_load_data=True, should_load_preprocessed=False,
                                                    #  should_load_data=True, should_load_preprocessed=True,
													)

In [5]:
most_recent_modern_found_EEG_recording_file_df['src_file'].to_list()
demo_xdf_paths: List[Path] = [Path(v) for v in most_recent_modern_found_EEG_recording_file_df['src_file'].to_list()]

modern_found_EEG_recording_file_df = most_recent_modern_found_EEG_recording_file_df

['E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T194023.711Z_eeg.xdf',
 'E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T134325.495Z_eeg.xdf',
 'E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T023809.607Z_eeg.xdf',
 'E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_rMBPPinkDot_2026-01-22T022657.502Z.xdf',
 'E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T222456.756Z_eeg.xdf',
 'E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T222210.952Z_eeg.xdf',
 'E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T221511.412Z_eeg.xdf',
 'E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T07014

In [6]:
modern_found_EEG_recording_file_df.sort_values(by='meas_datetime', ascending=False, inplace=False, ignore_index=True, na_position='last')

,src_file_name,start_t,src_file,meas_datetime,ctime,size,mtime
0,LabRecorder_Apogee_2026-01-22T194023.711Z_eeg,2026-01-22 19:40:23,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-22 19:40:23+00:00,2026-01-22 19:40:23.712537600,48476019,2026-01-22 20:25:26.173330944
1,LabRecorder_Apogee_2026-01-22T134325.495Z_eeg,2026-01-22 13:43:25,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-22 13:43:25+00:00,2026-01-22 13:43:25.497702400,48249985,2026-01-22 14:28:15.394905088
2,LabRecorder_Apogee_2026-01-22T023809.607Z_eeg,2026-01-22 02:38:09,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-22 02:38:09+00:00,2026-01-22 02:38:09.609662208,8747686,2026-01-22 02:53:27.174798080
3,LabRecorder_rMBPPinkDot_2026-01-22T022657.502Z,2026-01-22 02:26:57,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-22 02:26:57+00:00,2026-01-22 05:18:30.762354944,15192194,2026-01-22 02:53:46.000000000
4,LabRecorder_Apogee_2026-01-21T222456.756Z_eeg,2026-01-21 22:24:56,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-21 22:24:56+00:00,2026-01-21 22:24:56.756452864,75856723,2026-01-21 23:35:22.349929472
5,LabRecorder_Apogee_2026-01-21T222210.952Z_eeg,2026-01-21 22:22:10,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-21 22:22:10+00:00,2026-01-21 22:22:10.953361408,1227082,2026-01-21 22:23:22.686732032
6,LabRecorder_Apogee_2026-01-21T221511.412Z_eeg,2026-01-21 22:15:11,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-21 22:15:11+00:00,2026-01-21 22:15:11.419858432,6921285,2026-01-21 22:21:43.269472768
7,LabRecorder_Apogee_2026-01-21T070140.713Z_eeg,2026-01-21 07:01:40,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-21 07:01:40+00:00,2026-01-21 07:01:40.718356736,33453179,2026-01-21 07:32:46.056764160
8,LabRecorder_Apogee_2026-01-20T014745.931Z_eeg,2026-01-20 01:47:45,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-20 01:47:45+00:00,2026-01-20 01:47:45.934224896,124101249,2026-01-20 03:42:58.426438912
9,LabRecorder_Apogee_2026-01-19T204038.425Z_eeg,2026-01-19 20:40:38,E:/Dropbox (Personal)/Databases/UnparsedData/L...,2026-01-19 20:40:38+00:00,2026-01-19 20:40:38.429313280,85159355,2026-01-19 21:59:47.339383040


In [8]:
most_recent_xdfs = [Path(v).resolve() for v in deepcopy(modern_found_EEG_recording_file_df).head(n_most_recent_sessions_to_preprocess)['src_file'].tolist()]
most_recent_xdfs

[WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T194023.711Z_eeg.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T134325.495Z_eeg.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-22T023809.607Z_eeg.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_rMBPPinkDot_2026-01-22T022657.502Z.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T222456.756Z_eeg.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T222210.952Z_eeg.xdf'),
 WindowsPath('E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2026-01-21T221511.412Z_eeg.xdf'),
 WindowsPath('E:/Dr

In [ ]:
# modern_found_EEG_recording_files = HistoricalData.get_recording_files(recordings_dir=eeg_recordings_file_path)
# modern_found_EEG_recording_file_df: pd.DataFrame = HistoricalData.build_file_comparison_df(recording_files=modern_found_EEG_recording_files)

In [ ]:
# updated_file_paths, (pending_updated_recording_file_df, modern_found_EEG_recording_file_df, pre_processed_EEG_recording_file_df) = HistoricalData.discover_updated_recording_files(eeg_recordings_file_path=sso.eeg_recordings_file_path,
#                                                                                                                                                                                    eeg_analyzed_parent_export_path=sso.eeg_analyzed_parent_export_path)


In [9]:
# included_xdf_file_names = [
# 	"E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-21T051157.400Z_eeg.xdf", ## When it started to work
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-20T215045.162Z_eeg.xdf"
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-20T164055.381Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-18T092615.398Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-17T215112.606Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-17T124127.644Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-17T214946.083Z_eeg.xdf",
#     # # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-09-22T182649.051Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-16T220233.548Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-16T212744.771Z_eeg.xdf",
#     # # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-16T212721.939Z_eeg.xdf",
#     # # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-10-16T212528.076Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-09-23T141026.412Z_eeg.xdf",
#     # "E:/Dropbox (Personal)/Databases/UnparsedData/LabRecorderStudies/sub-P001/LabRecorder_Apogee_2025-09-22T213547.659Z_eeg.xdf",
# ]

included_xdf_file_names = deepcopy(most_recent_xdfs)

included_xdf_file_names = [Path(v).resolve() for v in included_xdf_file_names]
included_xdf_file_names = [v.name for v in included_xdf_file_names]


# included_xdf_file_names = None ## include all 
included_xdf_file_names

['LabRecorder_Apogee_2026-01-22T194023.711Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-22T134325.495Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-22T023809.607Z_eeg.xdf',
 'LabRecorder_rMBPPinkDot_2026-01-22T022657.502Z.xdf',
 'LabRecorder_Apogee_2026-01-21T222456.756Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-21T222210.952Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-21T221511.412Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-21T070140.713Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-20T014745.931Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-19T204038.425Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-19T125729.324Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-17T015049.800Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-16T140335.434Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-15T233747.758Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-15T231753.812Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-15T061015.990Z_eeg.xdf',
 'LabRecorder_Apogee_2026-01-14T145251.359Z_eeg.xdf']

# 2025-09-18 - LabRecorder XDF Imports

In [ ]:
from phoofflineeeganalysis.analysis.EEG_data import EEGData
from phoofflineeeganalysis.analysis.MNE_helpers import DatasetDatetimeBoundsRenderingMixin, RawArrayExtended, RawExtended, up_convert_raw_objects, up_convert_raw_obj
from phoofflineeeganalysis.analysis.xdf_files import LabRecorderXDF, XDFDataStreamAccessor


labRecorder_PostProcessed_path: Path = sso.eeg_analyzed_parent_export_path.joinpath(f'LabRecorder_PostProcessed')
labRecorder_PostProcessed_path.mkdir(exist_ok=True)

should_load_full_file_data: bool = True
# should_load_full_file_data: bool = False
# should_write_final_merged_eeg_fif: bool = True
should_write_final_merged_eeg_fif: bool = False

fail_on_exception = False
# fail_on_exception = True

_out_eeg_raw, _out_xdf_stream_infos_df, lab_recorder_xdf_files = LabRecorderXDF.load_and_process_all(lab_recorder_output_path=lab_recorder_output_path, 
                                                                                                     labRecorder_PostProcessed_path=labRecorder_PostProcessed_path, 
                                                                                                     should_load_full_file_data=should_load_full_file_data, should_write_final_merged_eeg_fif=should_write_final_merged_eeg_fif,
                                                                                                     included_xdf_file_names=included_xdf_file_names, fail_on_exception=fail_on_exception)
xdf_dataset_indicies = np.unique(deepcopy(_out_xdf_stream_infos_df).reset_index(drop=False, inplace=False)['xdf_dataset_idx'].to_numpy())
n_unique_xdf_datasets: int = len(xdf_dataset_indicies)
print(f'n_unique_xdf_datasets: {n_unique_xdf_datasets}')
## 2m 30s

[autoreload of phoofflineeeganalysis.analysis.SavedSessionsProcessor failed: Traceback (most recent call last):
  File "c:\Users\pho\repos\EmotivEpoc\PhoOfflineEEGAnalysis\.venv\lib\site-packages\IPython\extensions\autoreload.py", line 274, in check
    superreload(m, reload, self.old_objects, self.shell)
  File "c:\Users\pho\repos\EmotivEpoc\PhoOfflineEEGAnalysis\.venv\lib\site-packages\IPython\extensions\autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "c:\Users\pho\repos\EmotivEpoc\PhoOfflineEEGAnalysis\.venv\lib\site-packages\IPython\extensions\autoreload.py", line 397, in update_generic
    update(a, b)
  File "c:\Users\pho\repos\EmotivEpoc\PhoOfflineEEGAnalysis\.venv\lib\site-packages\IPython\extensions\autoreload.py", line 330, in update_class
    old_obj = getattr(old, key)
AttributeError: 'types.GenericAlias' object has no attribute '__copy__'. Did you mean: '__bool__'?
]
C:\Users\pho\repos\EmotivEpoc\PhoOfflineEEGAnalysis\src\phoofflineeega

limiting to included_xdf_file_names: ['LabRecorder_Apogee_2026-01-22T194023.711Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-22T134325.495Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-22T023809.607Z_eeg.xdf', 'LabRecorder_rMBPPinkDot_2026-01-22T022657.502Z.xdf', 'LabRecorder_Apogee_2026-01-21T222456.756Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-21T222210.952Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-21T221511.412Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-21T070140.713Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-20T014745.931Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-19T204038.425Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-19T125729.324Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-17T015049.800Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-16T140335.434Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-15T233747.758Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-15T231753.812Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-15T061015.990Z_eeg.xdf', 'LabRecorder_Apogee_2026-01-14T145251.359Z_eeg.xdf']...
	limited to 17/201 files
trying to process XDF file 0/17:

In [ ]:
_out_eeg_raw[-1].annotations.to_data_frame('datetime')

In [ ]:
(1435.384078 / 60.0) # ~24 mins

(3884.001499 / 60.0) # ~65 mins


In [ ]:
_out_xdf_stream_infos_df: pd.DataFrame = XDFDataStreamAccessor.init_from_results(_out_xdf_stream_infos_df=_out_xdf_stream_infos_df, active_only_out_eeg_raws=_out_eeg_raw) # [_out_xdf_stream_infos_df['name'] == 'Epoc X']
_out_xdf_stream_infos_df

In [ ]:
# ## INPUTS to timeline: _out_xdf_stream_infos_df: pd.DataFrame, _out_eeg_raw, lab_recorder_xdf_files, xdf_dataset_indicies

# from phoofflineeeganalysis.analysis.UI.historical_data_timeline import (
#     TimelineWidget, VideoMetadataTrack, EEGRecordingTrack, 
#     MotionRecordingTrack, PhoLogTrack, WhisperTrack
# )

# timeline = TimelineWidget()
# timeline.add_tracks_from_xdf_streams(_out_xdf_stream_infos_df)
# timeline.show()

In [ ]:
# from phoofflineeeganalysis.analysis.UI.historical_data_timeline import (
#     TimelineWidget, VideoMetadataTrack, EEGRecordingTrack, 
#     MotionRecordingTrack, PhoLogTrack, WhisperTrack
# )

# timeline = TimelineWidget()
# timeline.add_tracks_from_xdf_streams(_out_xdf_stream_infos_df)
# timeline.show()

In [ ]:
## sort by 'recording_day_date'
from datetime import timedelta


_filtering_xdf_stream_infos_df = deepcopy(_out_xdf_stream_infos_df).reset_index(drop=True)
_all_xdf_filenames = list(set(_filtering_xdf_stream_infos_df['xdf_filename'].to_list())) ## set(...) to de-duplicate the list

# _out_xdf_stream_infos_df['recording_day_date']
is_eeg_stream = (_filtering_xdf_stream_infos_df['type'] == 'EEG') ## only EEG files
_filtering_xdf_stream_infos_df = _filtering_xdf_stream_infos_df[is_eeg_stream].reset_index(drop=True)
# _filtering_xdf_stream_infos_df
# is_sufficiently_long = [(_filtering_xdf_stream_infos_df['duration_sec'] > timedelta(seconds=30.0))]
# _filtering_xdf_stream_infos_df = _filtering_xdf_stream_infos_df[is_sufficiently_long].reset_index(drop=True)
_filtering_xdf_stream_infos_df = _filtering_xdf_stream_infos_df[(_filtering_xdf_stream_infos_df['duration_sec'] > timedelta(seconds=90.0))].reset_index(drop=True)

_filtering_xdf_stream_infos_df = _filtering_xdf_stream_infos_df.sort_values(by=['recording_datetime', 'created_at_dt', 'first_timestamp_dt', 'last_timestamp_dt'], ascending=True, inplace=False)
_filtering_xdf_stream_infos_df
# _out_xdf_stream_infos_df[is_sufficiently_long]


In [ ]:
_all_xdf_filenames

In [ ]:
included_xdf_filenames: List[str] = _filtering_xdf_stream_infos_df['xdf_filename'].to_list()
excluded_xdf_filenames: List[str] = list(set(_all_xdf_filenames) - set(included_xdf_filenames))
excluded_xdf_filenames

# included_xdf_filenames = ['LabRecorder_2025-09-10T153731.079Z_eeg.xdf',
#  'LabRecorder_2025-09-11T014154.084Z_eeg.xdf',
#  'LabRecorder_2025-09-11T101328.256Z_eeg.xdf',
#  'LabRecorder_2025-09-11T154549.460Z_eeg.xdf',
#  'LabRecorder_2025-09-12T220903.464Z_eeg.xdf',
#  'LabRecorder_2025-09-18T031842.989Z_eeg.xdf',
#  'LabRecorder_2025-09-18T121337.267Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-12T215537.892Z.xdf',
#  'LabRecorder_Apogee_2025-09-18T151839.043Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-18T152308.395Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-19T051346.012Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-19T205118.364Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-20T214749.964Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-21T003051.428Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-21T085541.696Z_eeg.xdf',
#  'LabRecorder_Apogee_2025-09-22T213547.659Z_eeg.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-12T014018.162Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-13T021042.451Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-13T031209.598Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-19T001746.449Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-19T012938.518Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-19T015439.979Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-19T022210.910Z.xdf',
#  'LabRecorder_rMBPPinkDot_2025-09-20T023803.932Z.xdf']




# ['xdf_filename', 'first_timestamp', 'last_timestamp', 'sample_count',
    #    'lab_recorder_xdf_file_idx', 'xdf_filename', 'proccessed_fif_filename',
    #    'proccessed_mat_filename', 'xdf_dataset_idx', 'recording_datetime',
    #    'recording_day_date', 'duration_sec']

In [ ]:
_out_xdf_stream_infos_df.reset_index(drop=True).groupby('xdf_dataset_idx').first()

In [ ]:
lab_recorder_xdf_files

In [ ]:


def extract_annotations_df(active_only_out_eeg_raws) -> pd.DataFrame:
    ## Extract comments/notes/annotations/etc from the outputs

    _extracted_comments = []
    ignored_comment_descriptions = ['BAD_motion', '']
    for a_raw in active_only_out_eeg_raws:
        an_annotations = a_raw.annotations
        if (an_annotations is not None) and (len(an_annotations) > 0):
            an_annotation_df = an_annotations.to_data_frame(time_format='datetime')
            an_annotation_df = an_annotation_df[np.logical_not(np.isin(an_annotation_df['description'], ignored_comment_descriptions))]
            _extracted_comments.append(an_annotation_df)
            # an_annotation_df


    extracted_comments_df: pd.DataFrame = pd.concat(_extracted_comments)
    extracted_comments_df = extracted_comments_df.rename(columns={'onset':'time', 'description':'text'}, inplace=False)
    return extracted_comments_df

annotations_df = extract_annotations_df(_out_eeg_raw)
annotations_df

In [ ]:
# from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.SpikeRasterWidgets.Spike2DRaster import Spike2DRaster, SynchronizedPlotMode

In [ ]:
_out_xdf_stream_infos_df

In [ ]:
len(_out_xdf_stream_infos_df)
_out_xdf_stream_infos_df.loc[0]

In [ ]:
_out_eeg_raw

# Run Batch Computations on `_out_eeg_raw`

# ACTIVE: refine the active raws and compute them

In [ ]:
from phoofflineeeganalysis.analysis.EEG_data import EEGComputations, EEGData
from phoofflineeeganalysis.PendingNotebookCode import batch_compute_all_eeg_datasets, render_all_spectograms_to_high_quality_pdfs, plot_all_spectograms
from phoofflineeeganalysis.PendingNotebookCode import plot_session_spectogram
## INPUTS: _out_eeg_raw
# Process only the last 5 datasets using 4 workers:
# limit_num_items: int = 150
limit_num_items: int = 5
active_only_out_eeg_raws, results = batch_compute_all_eeg_datasets(eeg_raws=_out_eeg_raw, limit_num_items=limit_num_items, max_workers = 4)

## OUTPUTS: active_only_out_eeg_raws, results
# 1m 19.8s for 25 sessions

In [ ]:
from phoofflineeeganalysis.analysis.xdf_files import LabRecorderXDF

num_sessions: int = len(results)
num_sessions

# xdf_stream_infos_df: pd.DataFrame = XDFDataStreamAccessor.init_from_results(_out_xdf_stream_infos_df=_out_xdf_stream_infos_df, active_only_out_eeg_raws=active_only_out_eeg_raws)
xdf_stream_infos_df: pd.DataFrame = XDFDataStreamAccessor.init_from_results(_out_xdf_stream_infos_df=_out_xdf_stream_infos_df, active_only_out_eeg_raws=active_only_out_eeg_raws)
# xdf_stream_infos_df: pd.DataFrame = XDFDataStreamAccessor.init_from_results(_out_xdf_stream_infos_df=_out_xdf_stream_infos_df[_out_xdf_stream_infos_df['name'] == 'Epoc X'], active_only_out_eeg_raws=active_only_out_eeg_raws)
xdf_stream_infos_df

In [ ]:
## Extract comments/notes/annotations/etc from the outputs (`active_only_out_eeg_raws`)

_extracted_comments = []
ignored_comment_descriptions = ['BAD_motion', '']
for a_raw in active_only_out_eeg_raws:
    an_annotations = a_raw.annotations
    if (an_annotations is not None) and (len(an_annotations) > 0):
        an_annotation_df = an_annotations.to_data_frame(time_format='datetime')
        an_annotation_df = an_annotation_df[np.logical_not(np.isin(an_annotation_df['description'], ignored_comment_descriptions))]
        _extracted_comments.append(an_annotation_df)
        # an_annotation_df


extracted_comments_df: pd.DataFrame = pd.concat(_extracted_comments)
extracted_comments_df = extracted_comments_df.rename(columns={'onset':'time', 'description':'text'}, inplace=False)
extracted_comments_df

In [ ]:
## OUTPUTS: active_only_out_eeg_raws, results
## OUTPUTS: extracted_comments_df: pd.DataFrame ## comments track

In [ ]:
active_only_out_eeg_raws

In [ ]:
results

## Merged specific 5-day span where I know cognition performance

In [ ]:
from phoofflineeeganalysis.PendingNotebookCode import build_merged


cog_categorical_cats = ['cog_bad', 'cog_poor', 'cog_okay', 'cog_good', 'cog_great']

day_status_dict = {
    # '2025-09-19/01-54-39': 'cog_poor',
    # '2025-09-18/15-23-08': 'cog_poor',
    # '2025-09-22/21-35-47': 'cog_okay',
    # '2025-09-19/20-51-18': 'cog_good',
    # '2025-09-18/03-18-42': 'cog_great',
    # '2025-09-22/21-35-47': 'cog_bad',
    # '2025-09-21/08-55-41': 'cog_bad',
    ## new sessions
    # '2025-11-06/15:02:52': 'cog_okay',
    '2025-11-10/19-28-05': 'cog_okay',
    '2025-11-11/22-22-22': 'cog_bad',

}


# combined_spectogram_da: xr.DataArray = build_merged(active_only_out_eeg_raws=active_only_out_eeg_raws, results=results, day_status_dict=day_status_dict)
# combined_spectogram_da

combined_spectogram_ds, combined_spectogram_da = build_merged(active_only_out_eeg_raws=active_only_out_eeg_raws, results=results, day_status_dict=day_status_dict)
combined_spectogram_ds

In [ ]:
# netcdf_save_path = Path("2025-10-16_saved_spectogram.nc")
outputs_root_folder: Path = Path(r'L:\AITEMP\PhoOfflineEEGAnalysisOutputs').resolve()
outputs_root_folder.exists()

# netcdf_save_path = Path("2025-10-16_saved_spectogram.nc")
netcdf_save_path: Path = outputs_root_folder.joinpath("2025-12-11_saved_spectogram.nc").resolve()
combined_spectogram_da.to_netcdf(netcdf_save_path)
print(f'netcdf_save_path: "{netcdf_save_path.as_posix()}"')
# combined_spectogram_da.to_
# combined_spectogram_ds.to_netcdf(

## Plotting/Viz

In [ ]:
from phoofflineeeganalysis.analysis.UI.standalone_data_browser_gui import make_app

## INPUTS: combined_spectogram_ds, extracted_comments_df
app = make_app(ds=combined_spectogram_ds, comments_df=extracted_comments_df)
pn.serve(app, show=True)

### 2025-09-30 - Programmatic output to .html interactive plot

In [ ]:
netcdf_save_path = Path(r'L:/AITEMP/PhoOfflineEEGAnalysisOutputs/2025-11-12_saved_spectogram.nc').resolve()
assert netcdf_save_path.exists()
ds_disk = xr.open_dataset(netcdf_save_path)
ds_disk = ds_disk.load()
ds_disk

ds_disk

 


In [ ]:
from phoofflineeeganalysis.analysis.UI.spectrogram_gui import make_app

app = make_app(ds_disk, channels_to_select=['AF3','F7','F3','FC5'])
# app = make_app(combined_spectogram_da, channels_to_select=['AF3','F7','F3','FC5'])
# app = make_app(combined_spectogram_ds, channels_to_select=['AF3','F7','F3','FC5'])
pn.serve(app, show=True) # if using programmatic server

# make_app(ds, channels_to_select)


# 2025-10-02 - Plot Single Raw EEG

In [ ]:
import holoviews as hv
import mne
import pandas as pd
import panel as pn
from holoviews.operation.downsample import downsample1d

# pn.extension('tabulator')
hv.extension('bokeh')
# hv.extension('plotly') # type: ignore

from bokeh.io import output_notebook, show
output_notebook()

from phoofflineeeganalysis.PendingNotebookCode import plot_holoviews_multichannel_raw

# then any show(fig) will display inline

a_raw = active_only_out_eeg_raws[-1]
a_raw
curves_overlay_lttb = plot_holoviews_multichannel_raw(a_raw=a_raw, time_col_name = 'time')
curves_overlay_lttb

## 2025-10-16 - Batch Figure Export to .html files

In [ ]:
from phoofflineeeganalysis.PendingNotebookCode import build_merged

## INPUTS: active_only_out_eeg_raws, results

cog_categorical_cats = ['cog_bad', 'cog_poor', 'cog_okay', 'cog_good', 'cog_great']

day_status_dict = {
    '2025-09-19/01-54-39': 'cog_poor',
    '2025-09-18/15-23-08': 'cog_poor',
    '2025-09-22/21-35-47': 'cog_okay',
    '2025-09-19/20-51-18': 'cog_good',
    '2025-09-18/03-18-42': 'cog_great',
    '2025-09-22/21-35-47': 'cog_bad',
    '2025-09-21/08-55-41': 'cog_bad',
}


# combined_spectogram_da: xr.DataArray = build_merged(active_only_out_eeg_raws=active_only_out_eeg_raws, results=results, day_status_dict=day_status_dict)
# combined_spectogram_da

combined_spectogram_ds, combined_spectogram_da = build_merged(active_only_out_eeg_raws=active_only_out_eeg_raws, results=results, day_status_dict=day_status_dict, only_include_sessions_with_status_entries=False)
combined_spectogram_ds

In [ ]:
## OUTPUTS: combined_spectogram_da
combined_spectogram_da.sel(session=['2025-09-10/15-37-31'])

In [ ]:
import holoviews as hv
from plotly.io import to_html
from bokeh.layouts import column, row
from bokeh.embed import file_html
from bokeh.resources import CDN
from phoofflineeeganalysis.PendingNotebookCode import plot_holoviews_multichannel_raw
from phoofflineeeganalysis.PendingNotebookCode import plot_scrollable_spectogram

# active_data = ds_disk
# # active_data = partial_sess_ds

channels_to_select = ['AF3', 'F3', 'AF4', 'F4']
# _out = plot_scrollable_spectogram(ds_disk=active_data,
#     channels_to_select = channels_to_select,
#     fig_export_path="spectrogram_sessions.html",
#     # fig_export_path=None,
#     plot_indiv_channels=False,
# )
# _out.show()



export_html_parent_path = Path(r"L:/SlowSwap/Databases/AnalysisData/MNE_preprocessed/outputs/viz").resolve()
assert export_html_parent_path.exists()

num_sessions: int = len(active_only_out_eeg_raws)
assert len(results) == num_sessions, f"len(results): {len(results)} != num_sessions: {num_sessions}, but they should be equal!"
print(f'exporting {num_sessions} sessions to "{export_html_parent_path.as_posix()}"....')
for idx, (a_raw, a_result) in enumerate(zip(active_only_out_eeg_raws, results)):
    a_meas_date = a_raw.info.get('meas_date')
    a_raw_key: str = a_meas_date.strftime("%Y-%m-%d/%H-%M-%S") # '2025-09-22/21-35-47' or '2025-09-10/15-37-31'
    print(f'raw[{idx}/{num_sessions}]: a_raw_key: "{a_raw_key}"')
    curves_overlay_lttb = plot_holoviews_multichannel_raw(a_raw=a_raw, time_col_name = 'time', enable_downsampling=False)
    print(f'\ttype(curves_overlay_lttb): {type(curves_overlay_lttb)}')
    active_data = combined_spectogram_da.sel(session=[a_raw_key]) ## retains the .sessions property
    a_scrollable_spectogram = plot_scrollable_spectogram(ds_disk=active_data,
        channels_to_select = channels_to_select,
        # fig_export_path="spectrogram_sessions.html",
        fig_export_path=None,
        plot_indiv_channels=False,
    )
    print(f'\ttype(a_scrollable_spectogram): {type(a_scrollable_spectogram)}')

    # --- Holoviews (Bokeh) HTML ---
    bokeh_obj = hv.render(curves_overlay_lttb, backend='bokeh')
    bokeh_html = file_html(bokeh_obj, resources=CDN, title=f"EEG Overlay {a_raw_key}")
    html_str: str = file_html(bokeh_obj, resources=CDN, title="EEG Overlay")


    # --- Plotly HTML ---
    plotly_html: str = to_html(a_scrollable_spectogram, include_plotlyjs='cdn', full_html=False)

    # --- Combine both into one page ---
    combined_html: str = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="utf-8">
        <title>EEG Session {idx}: {a_raw_key}</title>
        <style>
            body {{
                display: flex;
                flex-direction: column;
                gap: 2em;
                margin: 2em;
                font-family: sans-serif;
                background-color: #fafafa;
            }}
            iframe {{
                border: none;
                width: 100%;
                height: 1000px;
            }}
        </style>
    </head>
    <body>
        <h2>EEG Session {idx}: {a_raw_key}</h2>
        <section>
            <h3>Raw Multichannel Overlay</h3>
            {bokeh_html}
        </section>
        <section>
            <h3>Scrollable Spectrogram</h3>
            {plotly_html}
        </section>
    </body>
    </html>
    """

    ## TODO: how do I create a column of the two valid html strings?
    # all_items = hv.Div((curves_overlay_lttb, a_scrollable_spectogram))
    # convert Holoviews object to Bokeh
    # bokeh_obj = hv.render(curves_overlay_lttb, backend='bokeh')
    # html_str: str = file_html(bokeh_obj, resources=CDN, title="EEG Overlay")

    # write to HTML
    an_export_path = export_html_parent_path.joinpath(f"curves_overlay_lttb_{idx}.html").resolve()
    print(f'\twriting to "{an_export_path.as_posix()}"...')
    
    with open(an_export_path, "w", encoding="utf-8") as f:
        f.write(combined_html)

print(f'done with all.')

# XXX  2025-10-02 - tried to get it computing spectogram for Epoch-type objects in addition to RawEEG

In [ ]:
from phoofflineeeganalysis.analysis.EEG_data import EEGComputations, EEGData
from phoofflineeeganalysis.PendingNotebookCode import batch_compute_all_eeg_datasets, render_all_spectograms_to_high_quality_pdfs, plot_all_spectograms
from phoofflineeeganalysis.PendingNotebookCode import plot_session_spectogram

## INPUTS: _out_eeg_raw
## INPUTS: epochs_cleaned
epochs_cleaned_result = EEGComputations.run_all(raw=epochs_cleaned)

In [ ]:
import scipy.ndimage
from scipy.signal import spectrogram

## INPUTS: epochs_cleaned
# nperseg = 1024
# noverlap = 512

nperseg = 32
noverlap = 16

picks = mne.pick_types(epochs_cleaned.info, eeg=True, meg=False)

fs: float = epochs_cleaned.info["sfreq"]
data = epochs_cleaned.get_data(picks=picks) # np.shape(data) (229, 14, 384) - (n_epochs, n_chan, n_times)
ch_names = deepcopy(epochs_cleaned.info.ch_names)
# raw = deepcopy(raw)
# raw.down_convert_to_base_type()
Sxx_list = []
Sxx_avg_list = []

spectogram_result_dict = {}
for ch_idx, a_ch in enumerate(epochs_cleaned.info.ch_names):
    f, t, Sxx = spectrogram(data[ch_idx], fs=fs, nperseg=nperseg, noverlap=noverlap) # #TODO 2025-09-28 13:25: - [ ] Convert to newer `ShortTimeFFT.spectrogram`
    # np.shape(Sxx): (14, 17, 23) - (n_chan, n_freq, n_time)

    spectogram_result_dict[a_ch] = (f, t, Sxx) ## a tuple
    Sxx_list.append(Sxx) # np.shape(Sxx) # (513, 1116) - (n_freqs, n_times)
    Sxx_avg = np.nanmean(Sxx, axis=-1) ## average over all time to get one per session
    Sxx_avg_list.append(Sxx_avg)

Sxx_avg_list = np.stack(Sxx_avg_list) # (14, 513) - (n_channels, n_freqs)
Sxx_list = np.stack(Sxx_list) # (14, 513, 1116) - (n_channels, n_freqs, n_times)
    


In [ ]:
np.shape(f)
np.shape(t)
np.shape(Sxx) # (14, 17, 23) - (n_chan, n_freq, n_time)
np.shape(data[ch_idx]) # (14, 384) - (n_chan, n_epochs)

In [ ]:
np.shape(data) # (229, 14, 384) - (


In [ ]:
np.shape(Sxx_avg_list) # (14, 14, 17)
np.shape(Sxx_list) # (14, 14, 17, 23)

In [ ]:
Sxx_avg_list = xr.DataArray(Sxx_avg_list, dims=("channels", "freqs"), coords={"channels": ch_names, "freqs": f})


In [ ]:
Sxx_list = xr.DataArray(Sxx_list, dims=("channels", "freqs", "times"), coords={"channels": ch_names, "freqs": f, "times": t})


In [ ]:
combined_spectogram_ds, combined_spectogram_da = build_merged(active_only_out_eeg_raws=active_only_out_eeg_raws, results=results, day_status_dict=day_status_dict)
combined_spectogram_ds



In [ ]:
epochs.compute_psd().plot_topomap(ch_type="eeg", normalize=False, contours=0)


# 2025-10-01 - Plot Bokeh to .html file

In [ ]:
import holoviews as hv
from bokeh.embed import file_html
from bokeh.resources import CDN

# convert Holoviews object to Bokeh
bokeh_obj = hv.render(curves_overlay_lttb, backend='bokeh')

# write to HTML
html_str = file_html(bokeh_obj, CDN, "EEG Overlay")
with open("curves_overlay_lttb.html", "w", encoding="utf-8") as f:
    f.write(html_str)


# 2025-10-01 - Plot Single

In [ ]:
# ds_disk = 
partial_sess_ds = ds_disk.sel(session=['2025-09-21/08-55-41', '2025-09-22/21-35-47'])
partial_sess_ds

# small_netcdf_save_path = Path("2025-09-30_small_saved_spectogram.nc")
small_netcdf_save_path = Path("2025-11-05_small_saved_spectogram.nc")
partial_sess_ds.to_netcdf(small_netcdf_save_path)


In [ ]:
partial_sess_ds.cognitive_status.loc['2025-09-21/08-55-41'].item()

# 2025-10-02 - ONLY THING WORTH DOING IN THE WHOLE WORLD

In [ ]:
# a_raw.compute_psd().plot_topomap(ch_type="eeg", normalize=False, contours=0)

In [ ]:
a_raw.plot()

In [ ]:

from phoofflineeeganalysis.PendingNotebookCode import plot_scrollable_spectogram

active_data = ds_disk
# active_data = partial_sess_ds

channels_to_select = ['AF3', 'F3', 'AF4', 'F4']
_out = plot_scrollable_spectogram(ds_disk=active_data,
    channels_to_select = channels_to_select,
    fig_export_path="spectrogram_sessions.html",
    # fig_export_path=None,
    plot_indiv_channels=False,
)
_out.show()


## 2025-09-29 - Resume

In [ ]:
import hvplot.xarray

for a_result in results:
    spec_da = a_result['spectogram']['Sxx']
spec_da


In [ ]:
# spec_da = combined_spectogram_ds.sel(session='2025-09-19/20-51-18', cognitive_status='cog_great')
spec_da = combined_spectogram_da.sel(session='2025-09-19/20-51-18')
spec_da

In [ ]:
# --- Define the channels you want to select ---
channels_to_select = ['AF3', 'F7', 'F3', 'FC5']

# --- Use .sel() to select the desired channels by label ---
selected_da = spec_da.sel(channels=channels_to_select)
# selected_da = spec_da.sel(channels=['AF3'])
selected_da

# --- Average over the 'channels' dimension ---
# This is the key operation
da_mean = spec_da.mean(dim='channels')
da_mean

In [ ]:

# Assuming 'spec_da' is your xarray DataArray with dimensions ('channels', 'freqs', 'times')

# Plot each channel's spectrogram in a vertical stack
stacked_plot = selected_da.hvplot(
    x='times',
    y='freqs',
    col='channels',  # This is the key change to create a stacked column plot
    cmap='viridis',
    clim=(selected_da.min(), selected_da.max()),
    width=1800,
    height=900,      # You may want to adjust the height of individual plots
    title='Spectrograms per Channel'
).opts(width=900)

# layout.opts(
#     opts.Curve( height=200, width=900, xaxis=None, line_width=1.50, color='red', tools=['hover']),
#     opts.Spikes(height=150, width=900, yaxis=None, line_width=0.25, color='grey')).cols(1)

# %output backend='bokeh'
stacked_plot

In [ ]:
# display graph in browser
# a bokeh server is automatically started
bokeh_server = pn.Row(stacked_plot).show(port=12345)


In [ ]:

# stop the bokeh server (when needed)
bokeh_server.stop()

In [ ]:
hv.extension('bokeh', 'matplotlib')
from bokeh.plotting import show

show(hv.render(stacked_plot))

In [ ]:
%%output backend='bokeh' fig='png'
stacked_plot

In [ ]:
# plt.show()
fig = plt.figure()
fig.show()

In [ ]:
!bokeh info

In [ ]:
display(stacked_plot)


In [ ]:
# Generate an interactive spectrogram with a slider for the time dimension
hv_plot_slider = spec_da.hvplot(
    x='times',
    y='freqs',
    cmap='viridis',
    clim=(spec_da.min(), spec_da.max()), # Set color limits
    width=800,
    height=400,
    groupby='times', # This is the key argument
    title='Interactive Spectrogram with hvPlot Slider'
)

hv_plot_slider


In [ ]:

# Save the linked layout to a self-contained HTML file
hv.save(
    stacked_plot,
    'holoviz_spectrogram.html',
    backend='bokeh',
    # The resources parameter can be used to control how JS/CSS are included,
    # with 'INLINE' being the most portable option.
    resources='INLINE'
)



In [ ]:
from phoofflineeeganalysis.analysis.EEG_data import EEGComputations
from phoofflineeeganalysis.analysis.xdf_files import LabRecorderXDF

hdf5_out_path: Path = Path('E:/Dropbox (Personal)/Databases/AnalysisData/MNE_preprocessed').joinpath('2025-09-29_all_HDF.h5').resolve()

hdf5_out_path.unlink(missing_ok=True)

LabRecorderXDF.to_hdf(active_only_out_eeg_raws=active_only_out_eeg_raws, results=results, xdf_stream_infos_df=xdf_stream_infos_df, file_path=hdf5_out_path, root_key='/')


In [ ]:
flat_annotations = []
for an_xdf_dataset_idx in np.arange(num_sessions):
    a_raw = active_only_out_eeg_raws[an_xdf_dataset_idx]
    a_meas_date = a_raw.info.get('meas_date')
    a_raw_key: str = a_meas_date.strftime("%Y-%m-%d/%H-%M-%S") # '2025-09-22/21-35-47'
    a_result = results[an_xdf_dataset_idx]
    # EEGComputations.to_hdf(a_result=a_result, file_path=hdf5_out_path, root_key=f'/result/{a_raw_key}')
    # EEGComputations.to_hdf(a_result=a_result, file_path=f, root_key=f'/result/{a_raw_key}')
    # EEGComputations.perform_write_to_hdf(a_result=a_result, f=f, root_key=f'/result/{a_raw_key}')
    # a_stream_info = deepcopy(_out_xdf_stream_infos_df).loc[an_xdf_dataset_idx]    
    # print(f'i: {i}, a_meas_date: {a_meas_date}, a_stream_info: {a_stream_info}\n\n')
    # print(f'i: {an_xdf_dataset_idx}, a_meas_date: {a_meas_date}')
    an_annotations_df = a_raw.annotations.to_data_frame(time_format='datetime')
    an_annotations_df = an_annotations_df[an_annotations_df['description'] != 'BAD_motion']
    an_annotations_df['xdf_dataset_idx'] = an_xdf_dataset_idx
    an_annotations_df['xdf_raw_key'] = a_raw_key
    flat_annotations.append(an_annotations_df)

    ## Add metadata to a_raw and a_result
    # 2025-09-18/03-18-42

flat_annotations = pd.concat(flat_annotations, ignore_index=True)
flat_annotations

In [ ]:
import xarray as xr

cog_categorical_cats = ['cog_bad', 'cog_poor', 'cog_okay', 'cog_good', 'cog_great']

day_status_dict = {'2025-09-19/01-54-39': 'cog_poor',
'2025-09-18/15-23-08': 'cog_poor',
'2025-09-22/21-35-47': 'cog_okay',
'2025-09-19/20-51-18': 'cog_good',
'2025-09-18/03-18-42': 'cog_great',
'2025-09-22/21-35-47': 'cog_bad',
'2025-09-21/08-55-41': 'cog_bad',
}



In [ ]:
netcdf_save_path = Path("2025-09-30_saved_spectogram.nc")
combined_spectogram_da.to_netcdf(netcdf_save_path)



In [ ]:
netcdf_save_path = Path("2025-09-30_saved_spectogram.nc")

ds_disk = xr.open_dataset(netcdf_save_path)
ds_disk = ds_disk.load()
ds_disk


In [ ]:
import xarray as xr
import zarr
import numcodecs
import numpy as np
from pathlib import Path
from typing import Union
from phoofflineeeganalysis.PendingNotebookCode import ZarrSerialization

cog_categorical_cats = ['cog_bad', 'cog_poor', 'cog_okay', 'cog_good', 'cog_great']

day_status_dict = {
    '2025-09-19/01-54-39': 'cog_poor',
    '2025-09-18/15-23-08': 'cog_poor',
    '2025-09-22/21-35-47': 'cog_okay',
    '2025-09-19/20-51-18': 'cog_good',
    '2025-09-18/03-18-42': 'cog_great',
    '2025-09-22/21-35-47': 'cog_bad',
    '2025-09-21/08-55-41': 'cog_bad',
}


zarr_out_path = Path("2025-09-30_all_sessions.zarr")

zarr_out_path = ZarrSerialization.save_sessions_as_zarr(active_only_out_eeg_raws=active_only_out_eeg_raws, results=results, day_status_dict=day_status_dict, out_path=zarr_out_path)

## Immediately re-load from file
all_data = ZarrSerialization.load_sessions_from_zarr(zarr_out_path, verbose=True)


# Export Spectogram XArray Spectograms to Interactive Plots

In [ ]:
import xarray as xr
import zarr
import panel as pn
import hvplot.xarray
import numpy as np
from pathlib import Path

# # # --- 1. Data Loading Function ---

# # --- 2. Setup the Interactive Dashboard ---
# ZARR_FILE_PATH = Path("2025-09-29_all_sessions.zarr")

# try:
#     all_data = load_sessions_from_zarr(ZARR_FILE_PATH)
#     all_data = deepcopy(
# except (FileNotFoundError, ValueError) as e:
#     pn.pane.Alert(f"Error loading data: {e}", alert_type='danger').servable()
# else:


In [ ]:
# all_data = results
# all_data = all_Sxx
all_data = combined_spectogram_da
## INPUTS: all_data

pn.extension(sizing_mode="stretch_width")

# --- 3. Create Interactive Widgets ---
# cog_statuses = sorted(np.unique(all_data.cognitive_status.values).tolist())
cog_statuses = sorted(np.unique([v.attrs['cognitive_status'] for v in all_data]).tolist())

channels = all_data.channels.values.tolist()

status_select = pn.widgets.Select(name='Cognitive Status', options=['All'] + cog_statuses, value='All')
session_select = pn.widgets.Select(name='Session')
channel_select = pn.widgets.Select(name='Channel', options=channels)

# --- 4. Dependent Widgets ---
@pn.depends(status_select.param.value, watch=True)
def _update_session_options(status):
    if status == 'All':
        sessions = sorted(all_data.session.values.tolist())
    else:
        filtered_data = all_data.where(all_data.cognitive_status == status, drop=True)
        sessions = sorted(filtered_data.session.values.tolist())
    session_select.options = sessions
    if sessions:
        session_select.value = sessions[0]

# --- 5. Plot Function ---
@pn.depends(session_select.param.value, channel_select.param.value)
def plot_spectrogram(session_key, channel):
    if not session_key or channel is None:
        return pn.pane.Markdown("### Please select a session and channel to view the spectrogram.")
    spec_da = all_data['Sxx'].sel(session=session_key, channels=channel).squeeze()
    spec_da_db = 10 * np.log10(spec_da + 1e-10)
    spec_da_db.attrs['long_name'] = 'Power (dB)'
    plot = spec_da_db.hvplot.quadmesh(
        x='times', y='freqs', cmap='viridis', rasterize=True, height=500,
        title=f"Session: {session_key} | Channel: {channel}", xlabel="Time (s)", ylabel="Frequency (Hz)",
        clabel="Power Spectral Density (dB)"
    ).opts(responsive=True)
    return plot

# --- 6. Dashboard Layout ---
dashboard = pn.template.FastListTemplate(
    site="EEG Spectrogram Viewer",
    title="Interactive Session Analysis",
    sidebar=[status_select, session_select, channel_select],
    main=[plot_spectrogram],
    accent_base_color="#4A90E2",
    header_background="#4A90E2",
)

dashboard.servable()


In [ ]:
from pyphocorehelpers.plotting.image_plotting_helpers import IMShowHelpers


def plot_matrix(xbin_edges, ybin_edges, matrix, ax, **kwargs):

    def setup_stable_axes_limits(xbins_edges, ybin_edges, ax):
        " manually sets the axis data limits to disable autoscaling given the xbins_edges/ybin_edges "
        # x == horizontal orientation:
        ax.set_xlim(left=xbins_edges[0], right=xbins_edges[-1])
        ax.set_ylim(bottom=ybin_edges[0], top=ybin_edges[-1])


    variable_value = matrix

    xmin, xmax, ymin, ymax = (xbin_edges[0], xbin_edges[-1], ybin_edges[0], ybin_edges[-1]) # the same for both orientations
    x_first_extent = (xmin, xmax, ymin, ymax) # traditional order of the extant axes
    # y_first_extent = (ymin, ymax, xmin, xmax) # swapped the order of the extent axes.
    main_plot_kwargs = {
        'cmap': 'viridis',
        'origin':'lower',
        'extent':x_first_extent,
        'aspect': 'auto',        
    }

    """
    Note that changing the origin while keeping everything else the same doesn't flip the direction of the yaxis labels despite flipping the yaxis of the data.
    """
    im_out = ax.imshow(variable_value, **main_plot_kwargs)
    setup_stable_axes_limits(xbin_edges, ybin_edges, ax)
    return im_out




In [ ]:
fig = plt.figure(layout="constrained", figsize=[9, 9], dpi=220, clear=True) # figsize=[Width, height] in inches.
long_width_ratio = 1
ax_dict = fig.subplot_mosaic(
    [
        ["ax_dumb", "ax_dumb_avg"],
        ["ax_good", "ax_good_avg"],
		
        # ["ax_dumb_avg"],
        # ["ax_good_avg"],
    ],
    # set the height ratios between the rows
    # set the width ratios between the columns
    width_ratios=[10, 1],
    sharey=True,
    gridspec_kw=dict(wspace=0, hspace=0.0) # `wspace=0`` is responsible for sticking the pf and the activity axes together with no spacing
)
fig.show()

In [ ]:
import napari

viewer = napari.Viewer(ndisplay=3)

# viewer.add_points(mov_avg_filtered, size=2, face_color='red', name='mov_avg_filtered')
# viewer.add_points(data2, size=2, face_color='green', name='dataset2')
# viewer.add_points(data3, size=2, face_color='blue', name='dataset3')



In [ ]:

named_sessions_dict = {'dumb': -2,
					   'good': -6,
}

# dumb_session_idx: int = -2

_out_layers = {}

for a_name, an_xdf_dataset_idx in named_sessions_dict.items():
    a_raw = active_only_out_eeg_raws[an_xdf_dataset_idx]
    a_meas_date: datetime = a_raw.info.get('meas_date')
    a_raw_key: str = a_meas_date.strftime("%Y-%m-%d/%H-%M-%S") # '2025-09-22/21-35-47'

    # an_annotations_df = a_raw.annotations.to_data_frame(time_format='datetime')
    # an_annotations_df = an_annotations_df[an_annotations_df['description'] != 'BAD_motion']

    # a_result = results[an_xdf_dataset_idx]
    # # a_stream_info = deepcopy(_out_xdf_stream_infos_df).loc[an_xdf_dataset_idx]    
    # Sxx = a_result['spectogram']['Sxx']
    # Sxx_avg = a_result['spectogram']['Sxx_avg']
    # # t = a_result['spectogram']['t']
    # # freqs = a_result['spectogram']['freqs']
    # # fs = a_result['spectogram']['fs']
    # freqs, t, _ = a_result['spectogram']['spectogram_result_dict']['AF3']
    
    # Sxx_avg_across_channel_avg = np.atleast_2d(Sxx_avg.mean(dim='channels', skipna=True))
    # # Sxx_avg_across_channel_avg
    
    # Sxx_across_channel_avg = Sxx.mean(dim='channels', skipna=True)
    # # Sxx_across_channel_avg

    # ax_label = f"ax_{a_name}"
    # ax_label_avg = f"ax_{a_name}_avg"
    # # np.shape(Sxx_across_channel_avg)
    
    # # xbin = deepcopy(t)
    # # ybin = deepcopy(freqs)
    # # xmin, xmax, ymin, ymax = (xbin[0], xbin[-1], ybin[0], ybin[-1])
    # # # xmin, xmax, ymin, ymax = (active_one_step_decoder.ybin[0], active_one_step_decoder.ybin[-1], active_one_step_decoder.xbin[0], active_one_step_decoder.xbin[-1]) # Reversed x and y axes, seems not good.
    # # extent = (xmin, xmax, ymin, ymax)
    
    # # ax_dict[ax_label].imshow(Sxx_avg_across_channel_avg.T)
    # # ax_dict[ax_label_avg].imshow(Sxx_across_channel_avg, extent=extent, origin='lower')
    
    # # fig, axs, plot_im_out = IMShowHelpers.final_x_vertical_plot_imshow(xbin_edges=np.arange(1), ybin_edges=freqs, matrix=Sxx_avg_across_channel_avg, ax=ax_dict[ax_label])
    # # ax_dict[ax_label].autoscale(False)
    # # fig, axs, plot_im_out = IMShowHelpers.final_x_horizontal_plot_imshow(xbin_edges=t, ybin_edges=freqs, matrix=Sxx_across_channel_avg, ax=ax_dict[ax_label])
    # im_out = plot_matrix(xbin_edges=t, ybin_edges=freqs, matrix=Sxx_across_channel_avg, ax=ax_dict[ax_label])
    # ax_dict[ax_label].set_ylabel(a_name)
    # # ax_dict[ax_label].autoscale(False)
    

    # # fig, axs, plot_im_out = IMShowHelpers.final_x_vertical_plot_imshow(xbin_edges=t, ybin_edges=freqs, matrix=Sxx_avg_across_channel_avg, ax=ax_dict[ax_label_avg])
    # avg_im_out = plot_matrix(xbin_edges=[0.0, 1.0], ybin_edges=freqs, matrix=Sxx_avg_across_channel_avg, ax=ax_dict[ax_label_avg])
    # # ax_dict[ax_label_avg].set_ylabel(f"{a_name}_avg")

    
    # napari_img_layer_kwargs = dict(
    #     # channel_axis=None,
    #     # rgb=None,
    #     colormap='yellow', #'bop_blue',
    #     # contrast_limits=None,
    #     gamma=0.02,
    #     interpolation2d='nearest',
    #     interpolation3d='linear',
    #     rendering='mip',
    #     depiction='volume',
    #     # iso_threshold=None,
    #     # attenuation=0.05,
    #     # name=None,
    #     # metadata=None,
    #     # scale=None,
    #     translate=None,
    #     # rotate=None,
    #     # shear=None,
    #     # affine=None,
    #     # opacity=1,
    #     blending='additive',
    #     visible=True,
    #     # multiscale=None,
    #     # cache=True,
    #     # plane=None,
    #     # experimental_clipping_planes=None,
    #     # custom_interpolation_kernel_2d=None,
    # )
    

    # curr_name = f"{a_name}"
    # _out_layers[curr_name] = viewer.add_image(Sxx, name=curr_name, **napari_img_layer_kwargs)
    # viewer.dims.axis_labels = Sxx.dims # ('channels', 'freqs', 'times')

    # curr_name = f"{a_name}_avg"
    # # napari_img_layer_kwargs['translate'] = [0.0, 1852.0]
    # napari_img_layer_kwargs['translate'] = [0.0, -2.0]
    # _out_avg_layer = viewer.add_image(Sxx_avg.T, name=curr_name, **napari_img_layer_kwargs) #  colormap='bop_blue', gamma=0.20, rendering='additive'
    # _out_layers[curr_name] = _out_avg_layer


In [ ]:
# _out_avg_layer.translate([512.0, 0.0])
# _out_avg_layer.set_translation([512.0, 0.0])
# _out_avg_layer.translate = [0.0, 513.0]
_out_avg_layer.translate = [0.0, 1852.0]
_out_avg_layer.translate

In [ ]:
_out_avg_layer.

In [ ]:
viewer.dims
viewer.layers

In [ ]:
from pyphoplacecellanalysis.GUI.Napari.napari_helpers import napari_extract_layers_info

# @function_attributes(short_name=None, tags=['napari', 'config'], input_requires=[], output_provides=[], uses=[], used_by=['napari_extract_layers_info'], creation_date='2024-08-12 08:54', related_items=[])
def extract_layer_info(a_layer):
    """ Extracts info as a dict from a single Napari layer. 
    by default Napari layers print like: `<Shapes layer 'Shapes' at 0x1635a1e8460>`: without any properties that can be easily referenced.
    This function extracts a dict of properties.

    from pyphoplacecellanalysis.GUI.Napari.napari_helpers import extract_layer_info

    """
    out_properties_dict = {}
    positioning = ['scale', 'translate', 'rotate', 'shear', 'affine', 'corner_pixels']
    visual = ['opacity', 'blending', 'visible', 'z_index', 'contrast_limits_range', '_colormap_name', 'gamma']
    # positioning = ['scale', 'translate', 'rotate', 'shear', 'affine']
    out_properties_dict['positioning'] = {}

    for a_property_name in positioning:
        out_properties_dict['positioning'][a_property_name] = getattr(a_layer, a_property_name)

    out_properties_dict['visual'] = {}
    for a_property_name in visual:
        try:
            out_properties_dict['visual'][a_property_name] = getattr(a_layer, a_property_name)
        except (AttributeError, ValueError, KeyError, TypeError) as e:
            print(f'failed to get property: "{a_property_name}" with error {e}')
            pass
        except Exception as e:
            raise

    return out_properties_dict


# @function_attributes(short_name=None, tags=['napari', 'config'], input_requires=[], output_provides=[], uses=['extract_layer_info'], used_by=[], creation_date='2024-08-12 08:54', related_items=[])
def napari_extract_layers_info(layers):
	"""extracts info dict from each layer as well.
	Usage:
        from pyphoplacecellanalysis.GUI.Napari.napari_helpers import napari_extract_layers_info
		layers = directional_viewer.layers # [<Shapes layer 'Shapes' at 0x1635a1e8460>, <Shapes layer 'Shapes [1]' at 0x164d5402e50>]
		out_layers_info_dict = debug_print_layers_info(layers)

	"""
	out_layers_info_dict = {}
	for a_layer in layers:
		a_name: str = str(a_layer.name)
		out_properties_dict = extract_layer_info(a_layer)
		out_layers_info_dict[a_name] = out_properties_dict
		# if isinstance(a_layer, Shapes):
		# 	print(f'shapes layer: {a_layer}')
		# 	a_shapes_layer: Shapes = a_layer
		# 	# print(f'a_shapes_layer.properties: {a_shapes_layer.properties}')
		# 	out_properties_dict = extract_layer_info(a_layer)
		# 	print(f'\tout_properties_dict: {out_properties_dict}')
		# 	out_layers_info_dict[a_name] = out_properties_dict
		# else:
		# 	print(f'unknown layer: {a_layer}')	
	return out_layers_info_dict





layers = viewer.layers # [<Shapes layer 'Shapes' at 0x1635a1e8460>, <Shapes layer 'Shapes [1]' at 0x164d5402e50>]
out_layers_info_dict = napari_extract_layers_info(layers)
out_layers_info_dict

In [ ]:
out_layers_info_dict['dumb']

In [ ]:
from napari.layers.image.image import Image

property_names = ['metadata', 'blending', 'opacity', 'rendering', 'scale', 'gamma', 'contrast_limits_range', 'colormap']
for a_layer in layers:
    an_img_layer: Image = a_layer
    
#    type(a_layer)

# an_img_layer.blending
an_img_layer.__dict__

In [ ]:
napari.run()

In [ ]:
import xarray as xr

## INPUTS: a_result
a_spectogram_result: Dict = a_result['spectogram'] 

ch_names = a_spectogram_result['ch_names']
fs = a_spectogram_result['fs']
a_spectogram_result_dict = a_spectogram_result['spectogram_result_dict'] # Dict[channel: Tuple]
Sxx = a_spectogram_result['Sxx']
Sxx_avg = a_spectogram_result['Sxx_avg']

Sxx

In [ ]:

Sxx_avg_list = [] 

# ch_names = a_raw.info.ch_names

for a_ch, a_tuple in a_spectogram_result_dict.items():
    f, t, Sxx = a_tuple ## unpack the tuple
    # np.shape(Sxx) # (513, 1116) - (n_freqs, n_times)
    n_freqs = np.shape(f)
    n_times = np.shape(t) 
    Sxx_avg = np.nanmean(Sxx, axis=-1) ## average over all time to get one per session
    Sxx_avg_list.append(Sxx_avg)
    
Sxx_avg_list = np.stack(Sxx_avg_list) # (14, 513) - (n_channels, n_freqs)
Sxx_avg_list = xr.DataArray(Sxx_avg_list, dims=("channels", "freqs"), coords={"channels": ch_names, "freqs": f})
np.shape(Sxx_avg_list)
Sxx_avg_list

In [ ]:
ch_names

In [ ]:
a_raw.annotations

In [ ]:
Sxx_avg = np.nanmean(Sxx, axis=-1) ## average over all time to get one per session
Sxx_avg

In [ ]:
plt.close('all')

In [ ]:
plot_all_spectograms(active_only_out_eeg_raws, results)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["axes.titlesize"] = 8
plt.rcParams["axes.labelsize"] = 8
plt.rcParams["xtick.labelsize"] = 6
plt.rcParams["ytick.labelsize"] = 6
plt.rcParams["legend.fontsize"] = 6
plt.rcParams["figure.titlesize"] = 8
plt.rcParams["axes.titlepad"] = 0
plt.rcParams["figure.constrained_layout.use"] = True
plt.rcParams["figure.constrained_layout.h_pad"] = 0.0
plt.rcParams["figure.constrained_layout.w_pad"] = 0.0
plt.rcParams["figure.constrained_layout.hspace"] = 0.0
plt.rcParams["figure.constrained_layout.wspace"] = 0.0
plt.rcParams["figure.subplot.wspace"] = 0.0
plt.rcParams["figure.subplot.hspace"] = 0.0
plt.rcParams["figure.subplot.wspace"] = 0.0
plt.rcParams["figure.subplot.hspace"] = 0.0


In [ ]:
## Plot a synchronized EEG Raw data and Spectogram Figure together:
active_eeg_idx: int = -4
mne_raw_fig = active_only_out_eeg_raws[active_eeg_idx].plot(time_format='datetime', scalings='auto') # MNEBrowseFigure
fig, axs = plot_session_spectogram(active_only_out_eeg_raws[active_eeg_idx], results[active_eeg_idx], sync_to_mne_raw_fig=mne_raw_fig)
# plt.subplots_adjust(wspace=0, hspace=0)  # remove spacing


In [ ]:
active_eeg_idx: int = -6
mne_raw_fig2 = active_only_out_eeg_raws[active_eeg_idx].plot(time_format='datetime', scalings='auto') # MNEBrowseFigure
fig2, axs2 = plot_session_spectogram(active_only_out_eeg_raws[active_eeg_idx], results[active_eeg_idx], sync_to_mne_raw_fig=mne_raw_fig)

### Compute and show all spectograms

In [ ]:
# Compute results first
active_only_out_eeg_raws, results = batch_compute_all_eeg_datasets(eeg_raws=_out_eeg_raw, limit_num_items=3)

# Render to PDFs (paged)
from pathlib import Path
out_paths = render_all_spectograms_to_high_quality_pdfs(
    active_only_out_eeg_raws,
    results,
    output_parent_folder=Path(r"E:/Dropbox (Personal)/Databases/AnalysisData/MNE_preprocessed/exports"),
    mode="paged",
    seconds_per_page=180.0,
    freq_min_hz=1.0,
    freq_max_hz=40.0,
    dpi=300
)
print(f"Wrote {len(out_paths)} PDF(s)")

## ANalysis for Fatigue/Bandpowers

In [ ]:
from phoofflineeeganalysis.analysis.computations.fatigue_analysis import compare_multiple_recordings, compute_fatigue_metrics, analyze_fatigue_trends, print_analysis_report, visualize_fatigue_comparison

raw_objects = deepcopy(_out_eeg_raw)
raw_obj_labels = [str(a_raw) for a_raw in raw_objects]
if len(raw_objects) >= 2:
    results = compare_multiple_recordings(raw_objects, raw_obj_labels[:len(raw_objects)])
    print_analysis_report(results)
    visualize_fatigue_comparison(results)

    results
else:
    print("Not enough data files found for comparison.")


# 2025-12-10 - Build Timeline Browser from parsed streams

_out_xdf_stream_infos_df: pd.DataFrame = XDFDataStreamAccessor.init_from_results(_out_xdf_stream_infos_df=_out_xdf_stream_infos_df, active_only_out_eeg_raws=_out_eeg_raw) # [_out_xdf_stream_infos_df['name'] == 'Epoc X']
_out_xdf_stream_infos_df

In [ ]:
## INPUTS: _out_xdf_stream_infos_df: pd.DataFrame

In [ ]:
_out_xdf_stream_infos_df

# csv_save_path = Path('../output').joinpath('2025-12-10_all_xdf_stream_infos.csv').resolve()
csv_save_path = Path('../output').joinpath('2025-12-17_all_xdf_stream_infos.csv').resolve()
print(f'csv_save_path: "{csv_save_path.as_posix()}"')
_out_xdf_stream_infos_df.to_csv(csv_save_path)

## load with: 
# csv_save_path = Path('../output').joinpath('2025-12-10_all_xdf_stream_infos.csv').resolve()
# assert csv_save_path.exists()
# all_xdf_stream_infos_df: pd.DataFrame = pd.read_csv(csv_save_path)


In [ ]:
# from phoofflineeeganalysis.analysis.UI.historical_data_timeline import (
#     TimelineWidget, VideoMetadataTrack, EEGRecordingTrack, 
#     MotionRecordingTrack, PhoLogTrack, WhisperTrack
# )

timeline = TimelineWidget()


csv_save_path = Path('../output').joinpath('2025-12-09_parsed_videos.csv').resolve()
assert csv_save_path.exists()
video_df: pd.DataFrame = pd.read_csv(csv_save_path)

csv_save_path = Path('../output').joinpath('2025-12-10_all_xdf_stream_infos.csv').resolve()
assert csv_save_path.exists()
all_xdf_stream_infos_df: pd.DataFrame = pd.read_csv(csv_save_path)


# Add tracks from different modalities
timeline.add_track(VideoMetadataTrack(video_df))
timeline.add_tracks_from_xdf_streams(all_xdf_stream_infos_df)
timeline.show()

In [ ]:
timeline.add_tracks_from_xdf_streams(all_xdf_stream_infos_df)

In [ ]:

timeline.add_track(EEGRecordingTrack(eeg_df))
timeline.add_track(MotionRecordingTrack(motion_df))
timeline.add_track(PhoLogTrack(pho_log_df))
timeline.add_track(WhisperTrack(whisper_df))

# 2025-12-11 - Build Detailed Timeline Browser from actual recent data

In [ ]:
## INPUTS: _out_xdf_stream_infos_df: pd.DataFrame
from phoofflineeeganalysis.analysis.video_metadata import VideoMetadataParser

output_folder = Path('../output').resolve()
assert output_folder.exists()

# csv_save_path = output_folder.joinpath('2025-12-17_parsed_videos.csv').resolve()
# assert csv_save_path.exists()
# video_df: pd.DataFrame = pd.read_csv(csv_save_path)
# video_df


## parse videos from scratch because it's pretty fast:
video_recordings_folder_path = Path(r"M:\ScreenRecordings\EyeTrackerVR_Recordings")

print(f"Parsing videos in: {video_recordings_folder_path}")
video_df = VideoMetadataParser.parse_video_folder(video_recordings_folder_path)
video_df

In [ ]:
from phoofflineeeganalysis.analysis.UI.timeline.TimelineWidget import TimelineWidget
from phoofflineeeganalysis.analysis.UI.timeline import (
    TimelineWidget,
    TrackWidget,
    VideoMetadataTrack,
    EEGRecordingTrack,
    MotionRecordingTrack,
    PhoLogTrack,
    WhisperTrack,
    XDFStreamTrack,
    TrackRegistry,
)


timeline = TimelineWidget()

timeline.add_track(VideoMetadataTrack(video_df))
timeline.show()

In [ ]:
_out_xdf_stream_infos_df

csv_save_path = output_folder.joinpath('2025-12-15_all_xdf_stream_infos.csv').resolve()
print(f'csv_save_path: "{csv_save_path.as_posix()}"')
_out_xdf_stream_infos_df.to_csv(csv_save_path)

## load with: 
# csv_save_path = Path('../output').joinpath('2025-12-10_all_xdf_stream_infos.csv').resolve()
# assert csv_save_path.exists()
# all_xdf_stream_infos_df: pd.DataFrame = pd.read_csv(csv_save_path)


In [ ]:
csv_save_path = Path('../output').joinpath('2025-12-15_all_xdf_stream_infos.csv').resolve()
print(f'csv_save_path: "{csv_save_path.as_posix()}"')
_out_xdf_stream_infos_df.to_csv(csv_save_path)

In [ ]:
from phoofflineeeganalysis.analysis.UI.historical_data_timeline import (
    TimelineWidget, VideoMetadataTrack, EEGRecordingTrack, 
    MotionRecordingTrack, PhoLogTrack, WhisperTrack
)

timeline = TimelineWidget()


csv_save_path = Path('../output').joinpath('2025-12-09_parsed_videos.csv').resolve()
assert csv_save_path.exists()
video_df: pd.DataFrame = pd.read_csv(csv_save_path)

csv_save_path = Path('../output').joinpath('2025-12-10_all_xdf_stream_infos.csv').resolve()
assert csv_save_path.exists()
all_xdf_stream_infos_df: pd.DataFrame = pd.read_csv(csv_save_path)


# Add tracks from different modalities
timeline.add_track(VideoMetadataTrack(video_df))
timeline.add_tracks_from_xdf_streams(all_xdf_stream_infos_df)
timeline.show()

# From detailed data: 
INPUTS to timeline: _out_xdf_stream_infos_df: pd.DataFrame, _out_eeg_raw, lab_recorder_xdf_files, xdf_dataset_indicies


In [ ]:
## INPUTS: active_only_out_eeg_raws, results
## INPUTS: extracted_comments_df: pd.DataFrame ## comments track

In [ ]:
_out_xdf_stream_infos_df

In [ ]:
# lab_recorder_output_path

# print(list(_out_xdf_stream_infos_df.columns))

if 'xdf_file_path' not in _out_xdf_stream_infos_df.columns:
    _out_xdf_stream_infos_df['xdf_file_path'] = _out_xdf_stream_infos_df['xdf_filename'].map(lambda x: Path(lab_recorder_output_path).joinpath(x).resolve())


xdf_file_paths: List[Path] = _out_xdf_stream_infos_df['xdf_file_path'].to_list()
xdf_file_paths



In [ ]:
## INPUTS to timeline: _out_xdf_stream_infos_df: pd.DataFrame, _out_eeg_raw, lab_recorder_xdf_files, xdf_dataset_indicies
from phoofflineeeganalysis.analysis.xdf_files import LabRecorderXDF
from phoofflineeeganalysis.analysis.UI.timeline.datasource.datasources import XDFDatasource, DataframeDatasource, IntervalDataframeDatasource, BaseDatasource

a_ds = XDFDatasource(a_xdf_file=xdf_file_paths[0], datasource_name='test_xdf')
a_ds

In [ ]:
# a_ds.df
# list(a_ds.lab_recorder_xdf.stream_infos.columns)

a_ds.lab_recorder_xdf.stream_infos
# 'last_timestamp_dt'

In [ ]:
a_ds.total_datasource_start_end_times

In [ ]:

from phoofflineeeganalysis.analysis.UI.timeline import (
    TimelineWidget, VideoMetadataTrack, EEGRecordingTrack, 
    MotionRecordingTrack, PhoLogTrack, WhisperTrack, StringDataTrack, XDFStreamTrack, TrackWidget
)

## INPUTS: xdf_file_paths
timeline = TimelineWidget()

# Get stream information DataFrame from the datasource
stream_infos_df = a_ds.lab_recorder_xdf.stream_infos

# Add tracks for all available stream modalities
if stream_infos_df is not None and not stream_infos_df.empty:
    timeline.add_tracks_from_xdf_streams(stream_infos_df, fail_on_exception=False)

# Always add XDFStreamTrack in addition to individual modality tracks
timeline.add_track(track=XDFStreamTrack(a_ds))

timeline.show()

In [ ]:
a_ds.get_detailed_data()

In [ ]:
# a_ds.lab_recorder_xdf
a_ds.lab_recorder_xdf.datasets

In [ ]:
a_ds.lab_recorder_xdf.stream_infos

In [ ]:
from phoofflineeeganalysis.analysis.UI.timeline.tracks.MotionRecordingTrack import MotionRecordingTrack
from phoofflineeeganalysis.analysis.MNE_helpers import up_convert_raw_objects, up_convert_raw_obj
from phoofflineeeganalysis.analysis.MNE_helpers import MNEHelpers

## INPUTS: _out_xdf_stream_infos_df
a_motion_raw = up_convert_raw_objects(a_ds.lab_recorder_xdf.datasets_dict[DataModalityType.MOTION.value])[0]
a_motion_df = a_ds.lab_recorder_xdf.streams_timestamp_dfs['Epoc X Motion'] ## not right
a_motion_overview_df = a_ds.lab_recorder_xdf.stream_infos ## overview
dataset_MOTION_df = a_motion_raw.to_data_frame(time_format='datetime')
dataset_MOTION_df = MNEHelpers.convert_df_columns_to_datetime(dataset_MOTION_df, dt_col_names=["start_time", "end_time"])
dataset_MOTION_df
from phoofflineeeganalysis.analysis.UI.timeline.datasource.datasources import IntervalDataframeDatasource

motion_ds = IntervalDataframeDatasource(df=dataset_MOTION_df, time_column_name='time') # , time_col_name='time'
motion_ds
# def load_motion_series(metadata: Dict[str, Any], window_ts: Tuple[float, float]) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
#     # Use metadata['xdf_filename'], metadata['sampling_rate'], etc.
#     # Return: {"AccX": (t_accx, v_accx), "AccY": (...), ..., "GyroZ": (...)}
#     xdf_filepath = Path(metadata['xdf_filename']).resolve() ## load the detailed data from the XDF file
    
motion_track = MotionRecordingTrack(motion_source=motion_ds, height=80)
timeline.add_track(motion_track)

In [ ]:
motion_track._is_detailed_mode
# motion_track._ensure_detailed_items()
motion_track._render_detailed(motion_ds.total_datasource_start_end_times)
motion_track._is_detailed_mode

In [ ]:
motion_track._is_detailed_mode
motion_track.set_detailed_threshold(seconds=1000000.0)  # Adjust this value as needed
motion_track.update_display()
motion_track._is_detailed_mode

In [ ]:
motion_track._ensure_detailed_items()

In [ ]:
motion_ds.total_datasource_start_end_times
motion_ds.total_df_start_end_times


In [ ]:
timeline.set_time_range(start_dt=motion_ds.total_datasource_start_end_times[0], end_dt=motion_ds.total_datasource_start_end_times[1])

In [ ]:
# dataset_MOTION_df ## overview

# a_ds.lab_recorder_xdf.stream_infos ## overview
a_motion_raw[0].to_df()

## Other Tracks

In [ ]:
# Build the detailed PhoLogger track from `extracted_comments_df: pd.DataFrame` with columns: ['time', 'text']. It should componently layout the strings so they don't excessively overlap, elliding or wrapping when needed.
## It can use the full height to stagger strings that would otherwise overlap horizontally.
from phoofflineeeganalysis.analysis.UI.timeline import PhoLogTrack, WhisperTrack
## Extract comments/notes/annotations/etc from the outputs (`active_only_out_eeg_raws`)

_extracted_comments = []
ignored_comment_descriptions = ['BAD_motion', '']
for a_raw in active_only_out_eeg_raws:
    an_annotations = a_raw.annotations
    if (an_annotations is not None) and (len(an_annotations) > 0):
        an_annotation_df = an_annotations.to_data_frame(time_format='datetime')
        an_annotation_df = an_annotation_df[np.logical_not(np.isin(an_annotation_df['description'], ignored_comment_descriptions))]
        _extracted_comments.append(an_annotation_df)
        # an_annotation_df


extracted_comments_df: pd.DataFrame = pd.concat(_extracted_comments)
extracted_comments_df = extracted_comments_df.rename(columns={'onset':'time', 'description':'text'}, inplace=False)
extracted_comments_df

timeline.add_track(PhoLogTrack(extracted_comments_df))

In [ ]:
from phoofflineeeganalysis.analysis.MNE_helpers import DatasetDatetimeBoundsRenderingMixin, RawArrayExtended, RawExtended, up_convert_raw_objects, up_convert_raw_obj
from phoofflineeeganalysis.analysis.EEG_data import EEGData
from PhoOfflineEEGAnalysis.src.phoofflineeeganalysis.analysis.SavedSessionsProcessor import LabRecorderXDF


assert lab_recorder_output_path.exists()

lab_recorder_xdf_files: List[Path] = list(lab_recorder_output_path.glob('*.xdf'))
n_total_found_files: int = len(lab_recorder_xdf_files)
if included_xdf_file_names is not None:
    print(f'limiting to included_xdf_file_names: {included_xdf_file_names}...')
    lab_recorder_xdf_files = [v for v in lab_recorder_xdf_files if v.name in included_xdf_file_names]
    n_filtered_found_files: int = len(lab_recorder_xdf_files)
    print(f'\tlimited to {n_filtered_found_files}/{n_total_found_files} files')

if not should_load_full_file_data:
    assert (not should_write_final_merged_eeg_fif)

if (labRecorder_PostProcessed_path is not None) and should_write_final_merged_eeg_fif:
    labRecorder_PostProcessed_path.mkdir(exist_ok=True)

# a_xdf_file = lab_recorder_xdf_files[-3]
# a_xdf_file = lab_recorder_xdf_files[-1]
# a_xdf_file = Path(r"E:\Dropbox (Personal)\Databases\UnparsedData\LabRecorderStudies\sub-P001\LabRecorder_2025-09-18T031842.989Z_eeg.xdf").resolve()
# a_xdf_file = Path(r"E:\Dropbox (Personal)\Databases\UnparsedData\LabRecorderStudies\sub-P001\LabRecorder_2025-09-18T121337.267Z_eeg.xdf").resolve()

_out_eeg_raw = []
_out_xdf_stream_infos_df = []

for an_xdf_file_idx, a_xdf_file in enumerate(lab_recorder_xdf_files):
    print(f'trying to process XDF file {an_xdf_file_idx}/{len(lab_recorder_xdf_files)}: "{a_xdf_file.as_posix()}"...')
    try:
        _obj = LabRecorderXDF.init_from_lab_recorder_xdf_file(a_xdf_file=a_xdf_file, should_load_full_file_data=should_load_full_file_data, debug_print=True)
        stream_infos = _obj.stream_infos
        raws = _obj.datasets
        raws_dict = _obj.datasets_dict
        eeg_raws = raws_dict.get(DataModalityType.EEG.value, [])
        if len(eeg_raws) > 0:
            print(f'\tWARN: no EEG streams found in "{a_xdf_file.as_posix()}". Skipping file.')
            # Merge by device so we can handle multiple EEG streams per XDF
            merged_eeg_raws, merge_meta = LabRecorderXDF.merge_eeg_streams_by_device(
                eeg_raws=eeg_raws, strict_merge=False, debug_print=False
            )
            eeg_raw = up_convert_raw_obj(eeg_raw)
            EEGData.set_montage(datasets_EEG=[eeg_raw])
            eeg_raw.debug_test_annotations_timestamps()
            _out_eeg_raw.append(eeg_raw)
        
    except (ValueError, KeyError, AssertionError, TypeError) as e:
        print(f'\t failed with error: {e}\n\tskipping file.')
        if fail_on_exception:
            raise
        else:
            continue
        
    except Exception as e:
        print(f'\t failed with error: {e}\n\tskipping file.')
        raise



In [ ]:
motion_df = _out_xdf_stream_infos_df[_out_xdf_stream_infos_df['name'] == 'Epoc X Motion']
list(motion_df.columns) # 'xdf_filename', 'xdf_dataset_idx'
motion_df['xdf_filename']

In [ ]:
def load_motion_series(metadata: Dict[str, Any], window_ts: Tuple[float, float]) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    # Use metadata['xdf_filename'], metadata['sampling_rate'], etc.
    # Return: {"AccX": (t_accx, v_accx), "AccY": (...), ..., "GyroZ": (...)}
    xdf_filepath = Path(metadata['xdf_filename']).resolve() ## load the detailed data from the XDF file
    

DataModalityType.MOTION.value
## INPUTS: _out_xdf_stream_infos_df
motion_df = _out_xdf_stream_infos_df[_out_xdf_stream_infos_df['name'] == 'Epoc X Motion']
motion_track = MotionRecordingTrack(motion_df, detailed_data_provider=load_motion_series)
timeline.add_track(motion_track)

In [ ]:

timeline.add_track(EEGRecordingTrack(eeg_df))
timeline.add_track(MotionRecordingTrack(motion_df))
timeline.add_track(PhoLogTrack(pho_log_df))
timeline.add_track(WhisperTrack(whisper_df))

# 2025-01-05 - `pyPhoTimeline`

In [ ]:
from pypho_timeline.rendering.datasources.track_datasource import TrackDatasource, BaseTrackDatasource
from pypho_timeline.rendering.detail_renderers import PositionPlotDetailRenderer, VideoThumbnailDetailRenderer


class PositionTrackDatasource(BaseTrackDatasource):
    """Example TrackDatasource for position data.
    
    Inherits from BaseTrackDatasource and implements all required methods for
    displaying position data with async detail loading.
    """
    
    def __init__(self, position_df: pd.DataFrame, intervals_df: pd.DataFrame):
        """Initialize with position data and intervals.
        
        Args:
            position_df: DataFrame with columns ['t', 'x', 'y'] (or ['t', 'x'] for 1D)
            intervals_df: DataFrame with columns ['t_start', 't_duration'] for intervals
        """
        super().__init__()
        self.position_df = position_df
        self.intervals_df = intervals_df.copy()
        self.custom_datasource_name = "PositionTrack"
        
        # Add visualization columns to intervals
        self.intervals_df['series_vertical_offset'] = 0.0
        self.intervals_df['series_height'] = 1.0
        
        # Create pens and brushes
        color = pg.mkColor('blue')
        color.setAlphaF(0.3)
        pen = pg.mkPen(color, width=1)
        brush = pg.mkBrush(color)
        self.intervals_df['pen'] = [pen] * len(self.intervals_df)
        self.intervals_df['brush'] = [brush] * len(self.intervals_df)
    
    @property
    def df(self) -> pd.DataFrame:
        return self.intervals_df
    
    @property
    def time_column_names(self) -> list:
        return ['t_start', 't_duration', 't_end']
    
    @property
    def total_df_start_end_times(self) -> tuple:
        if len(self.intervals_df) == 0:
            return (0.0, 1.0)
        t_start = self.intervals_df['t_start'].min()
        t_end = (self.intervals_df['t_start'] + self.intervals_df['t_duration']).max()
        return (t_start, t_end)
    
    def get_updated_data_window(self, new_start: float, new_end: float) -> pd.DataFrame:
        """Get intervals overlapping with time window."""
        mask = (self.intervals_df['t_start'] + self.intervals_df['t_duration'] >= new_start) & \
               (self.intervals_df['t_start'] <= new_end)
        return self.intervals_df[mask].copy()
    
    def update_visualization_properties(self, dataframe_vis_columns_function):
        """Update visualization properties."""
        self.intervals_df = dataframe_vis_columns_function(self.intervals_df)
    
    def get_overview_intervals(self) -> pd.DataFrame:
        """Get overview intervals."""
        return self.intervals_df
    
    def fetch_detailed_data(self, interval: pd.Series) -> pd.DataFrame:
        """Fetch position data for an interval."""
        if self.position_df is None:
            return pd.DataFrame()  # Return empty DataFrame if no position data available
        t_start = interval['t_start']
        t_end = t_start + interval['t_duration']
        mask = (self.position_df['t'] >= t_start) & (self.position_df['t'] < t_end)
        return self.position_df[mask].copy()
    
    def get_detail_renderer(self):
        """Get detail renderer for position data."""
        if self.position_df is None:
            return PositionPlotDetailRenderer(pen_color='cyan', pen_width=2, y_column=None)
        return PositionPlotDetailRenderer(pen_color='cyan', pen_width=2, y_column='y' if 'y' in self.position_df.columns else None)
    
    def get_detail_cache_key(self, interval: pd.Series) -> str:
        """Get cache key for interval."""
        return f"position_{interval['t_start']:.3f}_{interval['t_duration']:.3f}"


class VideoTrackDatasource(BaseTrackDatasource):
    """Example TrackDatasource for video data.
    
    Inherits from BaseTrackDatasource and implements all required methods for
    displaying video intervals with async detail loading.
    """
    
    def __init__(self, video_intervals_df: pd.DataFrame):
        """Initialize with video intervals.
        
        Args:
            video_intervals_df: DataFrame with columns ['t_start', 't_duration', 'video_path']
        """
        super().__init__()
        self.video_intervals_df = video_intervals_df.copy()
        self.custom_datasource_name = "VideoTrack"
        
        # Add visualization columns
        self.video_intervals_df['series_vertical_offset'] = 0.0
        self.video_intervals_df['series_height'] = 50.0
        
        # Create pens and brushes
        color = pg.mkColor('green')
        color.setAlphaF(0.3)
        pen = pg.mkPen(color, width=1)
        brush = pg.mkBrush(color)
        self.video_intervals_df['pen'] = [pen] * len(self.video_intervals_df)
        self.video_intervals_df['brush'] = [brush] * len(self.video_intervals_df)
    
    @property
    def df(self) -> pd.DataFrame:
        return self.video_intervals_df
    
    @property
    def time_column_names(self) -> list:
        return ['t_start', 't_duration', 't_end']
    
    @property
    def total_df_start_end_times(self) -> tuple:
        if len(self.video_intervals_df) == 0:
            return (0.0, 1.0)
        t_start = self.video_intervals_df['t_start'].min()
        t_end = (self.video_intervals_df['t_start'] + self.video_intervals_df['t_duration']).max()
        return (t_start, t_end)
    
    def get_updated_data_window(self, new_start: float, new_end: float) -> pd.DataFrame:
        """Get intervals overlapping with time window."""
        mask = (self.video_intervals_df['t_start'] + self.video_intervals_df['t_duration'] >= new_start) & \
               (self.video_intervals_df['t_start'] <= new_end)
        return self.video_intervals_df[mask].copy()
    
    def update_visualization_properties(self, dataframe_vis_columns_function):
        """Update visualization properties."""
        self.video_intervals_df = dataframe_vis_columns_function(self.video_intervals_df)
    
    def get_overview_intervals(self) -> pd.DataFrame:
        """Get overview intervals."""
        return self.video_intervals_df
    
    def fetch_detailed_data(self, interval: pd.Series) -> dict:
        """Fetch video frames for an interval (simulated with random images)."""
        # In a real implementation, this would load video frames
        # For demo, generate synthetic frame data
        n_frames = max(1, int(interval['t_duration'] * 10))  # 10 fps
        frames = []
        for i in range(n_frames):
            # Generate a simple colored frame
            frame = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
            frames.append(frame)
        return {'frames': frames, 'timestamps': np.linspace(interval['t_start'], interval['t_start'] + interval['t_duration'], n_frames)}
    
    def get_detail_renderer(self):
        """Get detail renderer for video."""
        return VideoThumbnailDetailRenderer(thumbnail_height=50.0, spacing=0.1)
    
    def get_detail_cache_key(self, interval: pd.Series) -> str:
        """Get cache key for interval."""
        return f"video_{interval['t_start']:.3f}_{interval['t_duration']:.3f}"
